# שלב 04 — ניתוח מרכזיות (Centrality)

כל מדד מרכזיות שואל שאלה אחרת על 'מי חשוב ברשת':

- **Degree** — לכמה תחנות אפשר להגיע ישירות? (צפיפות מקומית)
- **Betweenness** — על כמה מסלולים קצרים התחנה יושבת? (תחנת מעבר / גשר)
- **PageRank** — חשיבות לפי איכות השכנים, לא רק כמותם.
- **Harmonic** — כמה התחנה קרובה לכל שאר הרשת? (נגישות)

רק שילוב של כולם נותן תמונה אמיתית.

In [ ]:
# התקנת הספריות הנדרשות (להריץ פעם אחת; אפשר לדלג אם כבר מותקנות)
%pip install pandas numpy networkx matplotlib seaborn python-bidi

In [ ]:
from pathlib import Path
import pickle, json
import pandas as pd
import numpy as np
import networkx as nx


def find_repo_root(start: Path) -> Path:
    for cand in [start.resolve(), *start.resolve().parents]:
        if (cand / "israel-public-transportation").exists():
            return cand
    raise FileNotFoundError("repo root not found - set ROOT manually")


ROOT = find_repo_root(Path.cwd())
BASE = ROOT / "public_transport_network_notebooks"
GRAPH_DIR = BASE / "outputs" / "02_graph_construction"
OUT_DIR = BASE / "outputs" / "04_centrality_analysis"
FIG_DIR = BASE / "figures" / "04_centrality_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
BETWEENNESS_K = 300   # מספר דגימות לקירוב Betweenness
print("GRAPH_DIR:", GRAPH_DIR)

In [ ]:
import re
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from bidi.algorithm import get_display


def _fix(t):
    """מסדר טקסט עברי לתצוגה נכונה (bidi). אנגלית ומספרים נשארים כמו שהם."""
    if isinstance(t, str) and any(0x590 <= ord(c) <= 0x5FF for c in t):
        return get_display(t)
    return t


import matplotlib.text as _mt
if not getattr(_mt.Text, "_bidi", False):
    _orig = _mt.Text.set_text
    def _set(self, s):
        if isinstance(s, str) and getattr(self, "_disp", None) == s:
            return _orig(self, s)
        f = _fix(s)
        if isinstance(f, str):
            self._disp = f
        return _orig(self, f)
    _mt.Text.set_text = _set
    _mt.Text._bidi = True

sns.set_theme(style="whitegrid", font_scale=1.1)
matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
print("עברית בגרפים מופעלת")

## טעינת הגרפים

קוראים את הגרף הלא מכוון (לרוב המדדים) ואת המכוון (ל‑PageRank ולדרגות נכנסות/יוצאות).

In [ ]:
def load_graph():
    with open(GRAPH_DIR / "graph_undirected.pkl", "rb") as f:
        G = pickle.load(f)
    with open(GRAPH_DIR / "graph_directed.pkl", "rb") as f:
        D = pickle.load(f)
    return G, D


def largest_component(G):
    nodes = max(nx.connected_components(G), key=len)
    return G.subgraph(nodes).copy()


G, D = load_graph()
print(f"גרף נטען: {G.number_of_nodes():,} צמתים")

## חישוב כל מדדי המרכזיות

Betweenness מדויק יקר מאוד ברשת ארצית, ולכן מחשבים אותו בקירוב על הרכיב הגדול עם דגימת 300 צמתים (seed=42 לשחזור). Harmonic מחושב גם הוא על הרכיב הגדול. שאר המדדים על כל הרשת. **התא הזה עשוי לקחת כמה דקות.**

In [ ]:
def compute_all_centrality(G, D):
    print("  Degree Centrality ...")
    deg_c = nx.degree_centrality(G)

    print("  In/Out Degree ...")
    in_deg = dict(D.in_degree())
    out_deg = dict(D.out_degree())

    print("  PageRank ...")
    pr = nx.pagerank(D, weight="weight", alpha=0.85)

    print(f"  Betweenness (k={BETWEENNESS_K} samples) ...")
    Gc = largest_component(G)
    k = min(BETWEENNESS_K, Gc.number_of_nodes())
    btw = nx.betweenness_centrality(Gc, k=k, seed=42, normalized=True)

    print("  Harmonic Centrality ...")
    harm = nx.harmonic_centrality(Gc)

    rows = []
    for n in G.nodes():
        rows.append({
            "stop_id": n,
            "stop_name": G.nodes[n].get("stop_name", ""),
            "region": G.nodes[n].get("region", ""),
            "metro": G.nodes[n].get("metro", ""),
            "lat": G.nodes[n].get("lat"),
            "lon": G.nodes[n].get("lon"),
            "degree": G.degree(n),
            "degree_centrality": round(deg_c.get(n, 0), 6),
            "in_degree": in_deg.get(n, 0),
            "out_degree": out_deg.get(n, 0),
            "pagerank": round(pr.get(n, 0), 8),
            "betweenness": round(btw.get(n, 0), 8),
            "harmonic": round(harm.get(n, 0), 4),
        })
    return pd.DataFrame(rows)


df = compute_all_centrality(G, D)
df.to_csv(OUT_DIR / "stop_metrics.csv", index=False, encoding="utf-8-sig")
print(f"  stop_metrics.csv: {len(df)} שורות")
df.nlargest(5, "betweenness")[["stop_name", "region", "degree", "betweenness", "pagerank"]]

## שמירת Top-15 וקורלציה בין המדדים

שומרים את 15 התחנות המובילות לכל מדד, ומחשבים קורלציית Spearman בין המדדים — כדי לראות שהם לא מודדים אותו דבר (אחרת לא היה טעם בכמה מדדים).

In [ ]:
corr = df[["degree", "pagerank", "betweenness", "harmonic"]].corr(method="spearman").round(4)
corr.to_csv(OUT_DIR / "centrality_correlation_spearman.csv", encoding="utf-8-sig")

for metric in ["degree", "betweenness", "pagerank", "harmonic"]:
    df.nlargest(15, metric).to_csv(OUT_DIR / f"top_{metric}.csv", index=False, encoding="utf-8-sig")

corr

## גרפים: Top-15 לכל מדד, מטריצת קורלציה, פיזור ומפה

In [ ]:
def plot_top_bar(df, col, title, fname, color):
    top = df.nlargest(15, col)[["stop_name", "stop_id", col]].copy()
    top["label"] = top["stop_name"].where(top["stop_name"] != "", top["stop_id"])
    top = top.sort_values(col)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(top["label"], top[col], color=color)
    ax.set_xlabel(col.replace("_", " ").title())
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150)
    plt.show()


plot_top_bar(df, "degree", "Top 15 תחנות לפי Degree", "top_degree_bar.png", "#2563eb")
plot_top_bar(df, "betweenness", "Top 15 תחנות לפי Betweenness", "top_betweenness_bar.png", "#dc2626")
plot_top_bar(df, "pagerank", "Top 15 תחנות לפי PageRank", "top_pagerank_bar.png", "#16a34a")
plot_top_bar(df, "harmonic", "Top 15 תחנות לפי Harmonic", "top_harmonic_bar.png", "#d97706")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(df[["degree", "pagerank", "betweenness", "harmonic"]].corr(method="spearman"),
            annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("קורלציית Spearman בין מדדי Centrality")
plt.tight_layout()
plt.savefig(FIG_DIR / "centrality_correlation_heatmap.png", dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(df["degree"], df["betweenness"], s=3, alpha=0.3, color="#2563eb")
axes[0].set_xlabel("Degree"); axes[0].set_ylabel("Betweenness"); axes[0].set_title("Degree vs. Betweenness")
axes[1].scatter(df["degree"], df["pagerank"], s=3, alpha=0.3, color="#16a34a")
axes[1].set_xlabel("Degree"); axes[1].set_ylabel("PageRank"); axes[1].set_title("Degree vs. PageRank")
plt.tight_layout()
plt.savefig(FIG_DIR / "centrality_scatter.png", dpi=150)
plt.show()